In [4]:
%run svm_eve.py

[INFO] Loading data...
[INFO] Extracting features... input shape: (2136, 1000)
[INFO] Features shape: (2136, 10)
[INFO] Saved test set (features) -> results/test_set_features_linear.npz
[INFO] Saved test set (features + raw) -> results/test_set_both_linear.npz
[INFO] Starting training with GridSearchCV...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[INFO] Training finished, time: 8.50 s
[INFO] Best parameters: {'svm__C': np.float64(0.052140082879996844), 'svm__kernel': 'linear'}
[INFO] Model saved to ../svm1/svm_model_linear.pkl
[INFO] Predicting on test set...
[INFO] Accuracy: 0.5584
[INFO] Classification report:
               precision    recall  f1-score   support

           0       0.54      0.65      0.59       211
           1       0.58      0.47      0.52       217

    accuracy                           0.56       428
   macro avg       0.56      0.56      0.56       428
weighted avg       0.56      0.56      0.55       428

[INFO] Macro F1: 0.5551
[INFO] 

In [6]:
import joblib
import numpy as np

# load saved pipeline
model = joblib.load("svm_model_linear.pkl")

In [7]:
d = np.load("offts3_by_ij.npz")

rows = [np.asarray(d[f"{i}_{j}"]) for i in range(1,8) for j in range(1,4)]
min_len = min(len(r) for r in rows)

M = np.vstack([r[:min_len] for r in rows])  # (21, min_len)

In [8]:
def extract_features(data):
    """Extract statistical features from time distributions.

    Supports:
    - 1D input:  (n_features,)
    - 2D input:  (n_samples, n_features)

    Returns:
    - If input is 1D: shape (10,)
    - If input is 2D: shape (n_samples, 10)
    """
    data = np.asarray(data)

    if data.ndim == 1:
        data = data[None, :]   # convert to shape (1, n_timepoints)
        squeeze_output = True
    elif data.ndim == 2:
        squeeze_output = False
    else:
        raise ValueError("Input data must be a 1D or 2D array.")

    n_features = data.shape[1]

    mean = np.mean(data, axis=1, keepdims=True)
    std = np.std(data, axis=1, keepdims=True)
    cv = std / (mean + 1e-10)  # avoid division by zero
    max_ = np.max(data, axis=1, keepdims=True)

    percentile25 = np.percentile(data, 25, axis=1, keepdims=True)
    percentile50 = np.percentile(data, 50, axis=1, keepdims=True)
    percentile75 = np.percentile(data, 75, axis=1, keepdims=True)

    prop_gt1 = np.sum(data > 1, axis=1, keepdims=True) / n_features
    prop_gt2 = np.sum(data > 2, axis=1, keepdims=True) / n_features
    prop_gt3 = np.sum(data > 3, axis=1, keepdims=True) / n_features

    features = np.hstack([
        mean, std, cv, max_,
        prop_gt1, prop_gt2, prop_gt3,
        percentile25, percentile50, percentile75
    ])

    if squeeze_output:
        return features[0]   # return shape (10,) for 1D input
    return features

In [9]:
y_preds = []
y_pred_probas = []

for i in range(21):   # row 0 to row 20
    fmat = extract_features(rows[i]).reshape(1, -1)
    y_pred = model.predict(fmat)
    y_pred_proba = model.predict_proba(fmat)

    y_preds.append(y_pred[0])               # scalar label
    y_pred_probas.append(y_pred_proba[0])   # 1D probability vector

y_preds = np.array(y_preds)
y_pred_probas = np.array(y_pred_probas)

In [8]:
y_pred_probas

array([[0.66620867, 0.33379133],
       [0.5388994 , 0.4611006 ],
       [0.67646513, 0.32353487],
       [0.9161637 , 0.0838363 ],
       [0.72509955, 0.27490045],
       [0.69426082, 0.30573918],
       [0.85215272, 0.14784728],
       [0.72560969, 0.27439031],
       [0.72121396, 0.27878604],
       [0.9585999 , 0.0414001 ],
       [0.79139241, 0.20860759],
       [0.81283711, 0.18716289],
       [0.93221016, 0.06778984],
       [0.81416176, 0.18583824],
       [0.74810193, 0.25189807],
       [0.86003399, 0.13996601],
       [0.80081702, 0.19918298],
       [0.6836952 , 0.3163048 ],
       [0.96384218, 0.03615782],
       [0.88947587, 0.11052413],
       [0.83245565, 0.16754435]])